# Pneumonia detection — Colab pipeline

A single-pass walkthrough of the project, framed as an **increasing-complexity ladder**.
Each rung adds one ingredient on top of the previous one and reports the held-out test accuracy.

| Rung | Setup | Adds vs. previous |
|------|-------|-------------------|
| 1 | ResNet50 from scratch (5-fold) | baseline — no transfer learning |
| 2 | ResNet50 + ImageNet pretrained (5-fold) | + transfer learning |
| 3 | + ConvNeXt-Tiny (10 models) | + architectural diversity |
| 4 | + SNR-AdamW ResNet50 (15 models) | + theory-driven optimizer (Litman & Guo 2026) |

The headline number is rung 4: the 15-model multi-architecture ensemble.

## Hardware
*Runtime → Change runtime type → T4 GPU* (or better).

## Time estimate (free T4)
- Setup + dataset download: ~3 min
- Step 1 — 5-fold ResNet50 from scratch: ~25 min
- Step 2 — 5-fold ResNet50 pretrained: ~25 min
- Step 3 — 5-fold ConvNeXt-Tiny: ~25 min
- Step 4 — 5-fold SNR-AdamW ResNet50: ~25 min
- Eval + plots: ~5 min
- **Total: ~110 min on a free T4** (well inside the ~12 h Colab session limit)

(The same pipeline takes ~30+ hours on AMD Vega 64 + DirectML on Windows.)

## Setup flow
Run sections 1-2, then **either** section 3 (recommended — uses Drive for credentials and persistent runs) **or** section 4 (manual upload). After that, run sections 5-12 in order.

## 1. Clone repo + verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
%cd /content
!git clone https://github.com/Cyberwookie69/Pneumonia-xray-training.git
%cd /content/Pneumonia-xray-training

## 2. Install dependencies

Colab already has PyTorch with CUDA. We just need the project's extras.

In [ ]:
!pip install -q timm grad-cam opencv-python-headless kaggle

## 3. Mount Drive + load Kaggle credentials (recommended)

Mounts Google Drive, then:
- Copies `kaggle.json` from `My Drive/kaggle.json` to `~/.kaggle/` if you've put it there (one-time setup — skip the upload widget every run).
- Sets `PNEUMONIA_RUNS` to a Drive folder so checkpoints survive the ~12 h Colab session timeout.

If `kaggle.json` is **not** on Drive, this cell still works (sets up persistence) and you fall through to section 4 to upload it manually.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# Pull kaggle.json from Drive if you've placed it there.
src = '/content/drive/MyDrive/kaggle.json'
kaggle_dir = os.path.expanduser('~/.kaggle')
if os.path.exists(src):
    os.makedirs(kaggle_dir, exist_ok=True)
    shutil.copy(src, os.path.join(kaggle_dir, 'kaggle.json'))
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
    print('✓ kaggle.json copied from Drive — section 4 can be skipped')
else:
    print(f'⚠ {src} not found — run section 4 to upload it manually,\n'
          f'  or place kaggle.json at that Drive path for next time.')

# Persist runs across Colab sessions.
os.makedirs('/content/drive/MyDrive/pneumonia_runs', exist_ok=True)
os.environ['PNEUMONIA_RUNS'] = '/content/drive/MyDrive/pneumonia_runs'
print(f"Runs will be saved to: {os.environ['PNEUMONIA_RUNS']}")

## 4. Kaggle authentication via upload widget (fallback)

Only needed if section 3 didn't find `kaggle.json` on your Drive. Click *Choose Files* and pick the `kaggle.json` you downloaded from https://www.kaggle.com/settings → "Create New API Token".

In [ ]:
import os
if os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    print('✓ kaggle.json already in place (loaded by section 3) — '
          'no need to upload')
else:
    from google.colab import files
    uploaded = files.upload()  # select kaggle.json
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 5. Download dataset (~2.3 GB, ~1 min)

In [ ]:
!python pneumonia.py

## 6. Step 1 — ResNet50 from scratch (baseline)

5-fold ResNet50 trained from random init (no ImageNet weights). This is the weakest rung — it isolates everything that pretraining gives us at Step 2. Run names: `scratch_f0..f4`.

In [ ]:
# No --pretrained flag → tag defaults to "scratch", produces scratch_f0..f4
!python pneumonia_run_folds.py --extra="--img_size 288 --num_workers 4"

## 7. Step 2 — + ImageNet pretraining (transfer learning)

Same architecture (ResNet50 @ 288), now initialised from ImageNet weights and fine-tuned with focal loss. The Step 1 → Step 2 delta is the contribution of transfer learning. Run names: `ens_f0..f4`.

In [ ]:
# --pretrained → tag defaults to "ens", produces ens_f0..f4
!python pneumonia_run_folds.py --pretrained --extra="--img_size 288 --num_workers 4"

## 8. Step 3 — + ConvNeXt-Tiny (architectural diversity)

A second pretrained backbone, 5-fold @ 224. Adding it to the ResNet50 ensemble brings the count to 10 models and tests whether *architectural* diversity helps on top of *fold* diversity. Run names: `cnx224_f0..f4`.

In [ ]:
# ConvNeXt-Tiny @ 224 — ~25 min on T4
# Paper: https://arxiv.org/abs/2201.03545
# A 2022 pure-CNN that matches Vision Transformers on ImageNet by borrowing
# their design choices (large kernels, LayerNorm, GELU, inverted bottleneck).
# Chosen here for architectural diversity vs. ResNet50 in the multi-arch ensemble.
!python pneumonia_run_folds.py --pretrained --extra="--model convnext_tiny.fb_in22k_ft_in1k" --tag cnx224

## 9. Step 4 — + SNR-AdamW ResNet50 (theory-driven optimizer)

A third 5-fold run, same backbone as Step 2 but trained with the SNR-gated AdamW optimizer (Litman & Guo 2026). Adding it brings the multi-arch ensemble to 15 models. Run names: `snr_r50_f0..f4`.

In [ ]:
# SNR-AdamW ResNet50 — ~25 min on T4
# Paper: https://arxiv.org/abs/2605.01172
# Adds a per-parameter signal-to-noise gate to AdamW: each step, updates are
# scaled by max(0, (μ²−σ²/(b−1))/(σ²+ε)) using EMA gradient mean μ and variance σ².
# Suppresses parameters whose minibatch SNR is below threshold; same wall-clock cost.
!python pneumonia_run_folds.py --pretrained --extra="--snr_optimizer" --tag snr_r50

## 10. Eval — the complexity ladder

Each rung is one row in the eventual report table. The 15-model multi-arch ensemble at the bottom is the headline number. Per-architecture references (ConvNeXt alone, SNR alone) are also printed for completeness.

In [ ]:
# === The complexity ladder (4 rungs) ===
print("\n=== Step 1 — ResNet50 from scratch (5-fold) ===")
!python pneumonia_eval.py --ensemble scratch_f0,scratch_f1,scratch_f2,scratch_f3,scratch_f4 --use_best --num_workers 0

print("\n=== Step 2 — ResNet50 pretrained (5-fold) ===")
!python pneumonia_eval.py --ensemble ens_f0,ens_f1,ens_f2,ens_f3,ens_f4 --use_best --num_workers 0

print("\n=== Step 3 — + ConvNeXt-Tiny (10 models) ===")
!python pneumonia_eval.py --ensemble ens_f0,ens_f1,ens_f2,ens_f3,ens_f4,cnx224_f0,cnx224_f1,cnx224_f2,cnx224_f3,cnx224_f4 --use_best --num_workers 0

print("\n=== Step 4 — + SNR-AdamW (15 models, multi-arch) — HEADLINE ===")
!python pneumonia_eval.py --ensemble ens_f0,ens_f1,ens_f2,ens_f3,ens_f4,cnx224_f0,cnx224_f1,cnx224_f2,cnx224_f3,cnx224_f4,snr_r50_f0,snr_r50_f1,snr_r50_f2,snr_r50_f3,snr_r50_f4 --use_best --num_workers 0

# === Per-architecture (reference, not on the ladder) ===
print("\n=== Reference: ConvNeXt-Tiny alone (5-fold) ===")
!python pneumonia_eval.py --ensemble cnx224_f0,cnx224_f1,cnx224_f2,cnx224_f3,cnx224_f4 --use_best --num_workers 0

print("\n=== Reference: SNR-AdamW ResNet50 alone (5-fold) ===")
!python pneumonia_eval.py --ensemble snr_r50_f0,snr_r50_f1,snr_r50_f2,snr_r50_f3,snr_r50_f4 --use_best --num_workers 0

## 11. Plots

In [ ]:
# Learning curves — one plot per rung
!python pneumonia_plots.py curves --runs scratch_f0,scratch_f1,scratch_f2,scratch_f3,scratch_f4
!python pneumonia_plots.py curves --runs ens_f0,ens_f1,ens_f2,ens_f3,ens_f4
!python pneumonia_plots.py curves --runs cnx224_f0,cnx224_f1,cnx224_f2,cnx224_f3,cnx224_f4
!python pneumonia_plots.py curves --runs snr_r50_f0,snr_r50_f1,snr_r50_f2,snr_r50_f3,snr_r50_f4

# t-SNE feature embedding + Grad-CAM (use the strongest single fold = ens_f0)
!python pneumonia_plots.py features --run ens_f0 --use_best
!python pneumonia_gradcam.py --run_name ens_f0 --use_best --n_samples 8

## 12. Display the figures inline

In [ ]:
from IPython.display import Image, display
import os

runs_root = os.environ.get('PNEUMONIA_RUNS', 'runs')
for relpath in [
    f'{runs_root}/plots/learning_curves_scratch.png',
    f'{runs_root}/plots/learning_curves_ens.png',
    f'{runs_root}/plots/learning_curves_cnx224.png',
    f'{runs_root}/plots/learning_curves_snr_r50.png',
    f'{runs_root}/plots/fold_best_val_ens.png',
    f'{runs_root}/ens_f0/plots/tsne_best.png',
]:
    if os.path.exists(relpath):
        print(relpath)
        display(Image(relpath))